# 01d — MCI Survival Cohort **Manual QC Packet** (Build 2, `v1`)

Produces a **reviewable packet for human validation** of anchor matching and outcome derivation on
the **frozen Build 1 cohort**.

## This build is REVIEW-ONLY
- It **reads** the Build 1 freeze; it **never** re-derives, re-selects or modifies the cohort.
- It **does not** exclude anyone — including the eight pre-anchor-dementia cases, which **remain in
  the frozen v1 cohort**.
- Code **selects cases and presents source context**. A human reviewer fills in every adjudication
  field. No QC field is auto-populated.
- **Warning flags are not exclusions.**
- Build 1 outputs are hashed **before and after** this notebook runs and asserted byte-identical.

## Mandatory adjudication stratum
Build 1 found **8 participants with ≥1 dementia diagnosis *before* their selected MCI plasma anchor**
(Dementia → reversion → MCI-at-anchor). They are selected **programmatically from the Build 1 warning
flag** `qc_dementia_on_or_before_anchor` — never from a hard-coded RID list — and all 8 are mandatory.

For an *incident* MCI→dementia study this matters: a post-anchor dementia code in such a participant
may be **re-documentation of prevalent dementia** rather than **incident progression**. The reviewer
decides; the code does not.

## 0. Setup — paths, seed, and a read-only guard on the Build 1 freeze

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


def find_project_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "Data" / "raw").is_dir():
            return cand
    raise FileNotFoundError("Could not locate Data/raw above %s" % start)


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "Data" / "raw"
FREEZE_DIR = PROJECT_ROOT / "outputs" / "01c_mci_survival_cohort_freeze"

RANDOM_SEED = 20260711          # deterministic sampling seed (master spec)
VERSION = "v1"
ALIGN_TOL_DAYS = 90             # locked; used ONLY to re-annotate, never to re-select
CORE_ASSAYS = ["ptau217", "abeta42", "abeta40", "nfl", "gfap"]
MISSING_SENTINELS = [-4, -5]
BOUNDARY_LO, BOUNDARY_HI = 70, 90       # stratum-2 priority band
RAPID_DAYS = 180                        # stratum-3 priority band

# Build 1 artifacts -- READ ONLY. Never written by this notebook.
BUILD1_FILES = [
    "mci_survival_anchor_cohort_v1.tsv",
    "mci_survival_followup_cohort_v1.tsv",
    "mci_survival_primary_cohort_v1.tsv",
    "mci_survival_exclusion_log_v1.tsv",
    "mci_survival_cohort_flow_v1.tsv",
    "mci_survival_freeze_provenance_v1.json",
]

ASSERTIONS: list[dict] = []


def check(name: str, ok, detail: str = "") -> None:
    ok = bool(ok)
    ASSERTIONS.append({"assertion": name, "passed": ok, "detail": detail})
    if not ok:
        raise AssertionError(f"ASSERTION FAILED: {name}\n  diagnostics: {detail}")
    print(f"  PASS  {name}" + (f"  [{detail}]" if detail else ""))


def rid_list(rids, limit: int = 20) -> str:
    rids = sorted(int(r) for r in rids)
    head = ", ".join(str(r) for r in rids[:limit])
    return f"n={len(rids)} RIDs=[{head}{', ...' if len(rids) > limit else ''}]"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def save_tsv(df: pd.DataFrame, name: str) -> Path:
    path = FREEZE_DIR / name
    assert name not in BUILD1_FILES, f"REFUSING to overwrite a Build 1 artifact: {name}"
    df.to_csv(path, sep="\t", index=False)
    print(f"  saved -> {path.relative_to(PROJECT_ROOT)}   ({df.shape[0]} x {df.shape[1]})")
    return path


# ---- hash the Build 1 freeze BEFORE doing anything ----
HASHES_BEFORE = {f: sha256(FREEZE_DIR / f) for f in BUILD1_FILES}
print("Build 1 freeze — hashes BEFORE Build 2:")
for f, h in HASHES_BEFORE.items():
    print(f"  {f:44s} {h[:16]}...")
print(f"\nseed={RANDOM_SEED} | python {sys.version.split()[0]} | pandas {pd.__version__}")

Build 1 freeze — hashes BEFORE Build 2:
  mci_survival_anchor_cohort_v1.tsv            64e66814726c2263...
  mci_survival_followup_cohort_v1.tsv          8203a70f7cc8d030...
  mci_survival_primary_cohort_v1.tsv           a15e2cb8606f676b...
  mci_survival_exclusion_log_v1.tsv            a3eafebcdd3e392b...
  mci_survival_cohort_flow_v1.tsv              39b88d1a9f6cc3e4...
  mci_survival_freeze_provenance_v1.json       d32c113a0406242b...

seed=20260711 | python 3.14.0 | pandas 3.0.1


## 1. Load the frozen Build 1 cohort  *(authoritative — never re-derived)*

The freeze manifest records a SHA-256 for every Build 1 output. We verify the files on disk still
match those recorded hashes, so the packet is provably built against the **frozen** cohort.

In [2]:
manifest = json.loads((FREEZE_DIR / f"mci_survival_freeze_provenance_{VERSION}.json").read_text())

BOOL_COLS = ["same_day_alignment", "followup_eligible", "primary_eligible", "broad_anchor_eligible",
             "qc_dementia_on_or_before_anchor", "entry_age_available", "apoe4_available",
             "ptau217_available", "gfap_available", "nfl_available", "amyloid_ratio_available",
             "age_is_entry_age_fallback", "age_derivation_suspect", "entry_age_has_datestamp",
             "nonpositive_followup_flag"]


def load_frozen(name: str) -> pd.DataFrame:
    d = pd.read_csv(FREEZE_DIR / name, sep="\t")
    for c in BOOL_COLS:
        if c in d.columns:
            d[c] = d[c].astype("boolean").fillna(False).astype(bool)
    for c in [c for c in d.columns if c.endswith("_date")]:
        d[c] = pd.to_datetime(d[c], errors="coerce")
    return d


anchor_frozen = load_frozen(f"mci_survival_anchor_cohort_{VERSION}.tsv")
followup_frozen = load_frozen(f"mci_survival_followup_cohort_{VERSION}.tsv")
primary_frozen = load_frozen(f"mci_survival_primary_cohort_{VERSION}.tsv")
exclusion_log = load_frozen(f"mci_survival_exclusion_log_{VERSION}.tsv")

# verify the freeze on disk still matches the manifest hashes recorded at Build 1
for rec in manifest["outputs"]:
    fn = Path(rec["file"]).name
    check(f"[freeze integrity] {fn} matches its Build 1 manifest SHA-256",
          sha256(FREEZE_DIR / fn) == rec["sha256"])

N_ANCHOR, N_FU, N_PRIMARY = len(anchor_frozen), len(followup_frozen), len(primary_frozen)
N_FU_EVENTS = int((followup_frozen["event_indicator"] == 1).sum())
N_PRIMARY_EVENTS = int((primary_frozen["event_indicator"] == 1).sum())
print(f"\nFrozen cohort: A={N_ANCHOR}  B={N_FU} ({N_FU_EVENTS} events)  C={N_PRIMARY} ({N_PRIMARY_EVENTS} events)")

# The mandatory stratum is derived FROM THE FLAG, never from a hard-coded RID list.
MANDATORY = anchor_frozen.loc[anchor_frozen["qc_dementia_on_or_before_anchor"], "RID"]
MANDATORY = np.sort(MANDATORY.to_numpy())
print(f"Mandatory pre-anchor-dementia participants (from Build 1 flag): n={len(MANDATORY)} -> {list(MANDATORY)}")

  PASS  [freeze integrity] mci_survival_anchor_cohort_v1.tsv matches its Build 1 manifest SHA-256
  PASS  [freeze integrity] mci_survival_followup_cohort_v1.tsv matches its Build 1 manifest SHA-256
  PASS  [freeze integrity] mci_survival_primary_cohort_v1.tsv matches its Build 1 manifest SHA-256
  PASS  [freeze integrity] mci_survival_exclusion_log_v1.tsv matches its Build 1 manifest SHA-256
  PASS  [freeze integrity] mci_survival_cohort_flow_v1.tsv matches its Build 1 manifest SHA-256

Frozen cohort: A=535  B=410 (86 events)  C=401 (85 events)
Mandatory pre-anchor-dementia participants (from Build 1 flag): n=8 -> [np.int64(467), np.int64(4115), np.int64(4506), np.int64(4706), np.int64(6976), np.int64(7070), np.int64(10251), np.int64(10322)]


## 2. Load raw source tables with the **same** deterministic source-row rule as Build 1

`<prefix>_src_row` = 0-based positional index of the record in the raw CSV as delivered, assigned
immediately after `pd.read_csv`. Reading *more* columns than Build 1 did cannot change these indices
(row order and row count are unchanged) — asserted below by round-tripping every frozen source-row id
back to the raw table and confirming the dates agree.

We additionally load the **diagnostic-confidence / adjudication** fields (`DXCONFID`, `DXNORM`,
`DXMCI`, `DXAD`), which the reviewer needs but Build 1 did not require.

In [3]:
def load_raw(fname: str, prefix: str, **kw) -> pd.DataFrame:
    df = pd.read_csv(RAW_DIR / fname, **kw)
    df[f"{prefix}_src_row"] = np.arange(len(df), dtype="int64")
    return df


DX_FILE = "All_Subjects_DXSUM_05Mar2026.csv"
PL_FILE = "All_Subjects_UPENN_PLASMA_FUJIREBIO_QUANTERIX_05Mar2026.csv"

dx_raw = load_raw(DX_FILE, "dx", usecols=[
    "ID", "PTID", "RID", "VISCODE2", "EXAMDATE", "DIAGNOSIS",
    "DXNORM", "DXMCI", "DXAD", "DXCONFID"]).rename(columns={"ID": "dx_src_id"})
pl_raw = load_raw(PL_FILE, "plasma", usecols=[
    "PTID", "RID", "VISCODE2", "EXAMDATE",
    "pT217_F", "AB42_F", "AB40_F", "AB42_AB40_F", "NfL_Q", "GFAP_Q", "Comment"])

dx_raw["raw_EXAMDATE"] = dx_raw["EXAMDATE"].astype("string")
pl_raw["raw_EXAMDATE"] = pl_raw["EXAMDATE"].astype("string")
dx_raw["DATE"] = pd.to_datetime(dx_raw["EXAMDATE"], errors="coerce")
pl_raw["DATE"] = pd.to_datetime(pl_raw["EXAMDATE"], errors="coerce")
dx_raw["dx_code"] = pd.to_numeric(dx_raw["DIAGNOSIS"], errors="coerce")
dx_raw["dx_label"] = dx_raw["dx_code"].map({1.0: "CN", 2.0: "MCI", 3.0: "Dementia"}).fillna("unknown/other")

print(f"raw DXSUM rows: {len(dx_raw)} | raw plasma rows: {len(pl_raw)}")

# ---- ASSERTION 5 (part 1): frozen source-row ids round-trip to the raw tables ----
_dxi = dx_raw.set_index("dx_src_row")
_pli = pl_raw.set_index("plasma_src_row")

_a = anchor_frozen
_bad = _a[_a["anchor_src_row"].map(_pli["DATE"]) != _a["anchor_date"]]
check("[trace] every frozen anchor_src_row resolves to the raw plasma row with the same draw date",
      _bad.empty, rid_list(_bad["RID"]))
_bad = _a[_a["anchor_dx_src_row"].map(_dxi["DATE"]) != _a["anchor_dx_date"]]
check("[trace] every frozen anchor_dx_src_row resolves to the raw DXSUM row with the same dx date",
      _bad.empty, rid_list(_bad["RID"]))

_ev = _a[_a["event_indicator"] == 1]
_bad = _ev[_ev["event_src_row"].map(_dxi["DATE"]) != _ev["event_date"]]
check("[trace] every frozen event_src_row resolves to the raw DXSUM row with the same event date",
      _bad.empty, rid_list(_bad["RID"]))
_bad = _ev[_ev["event_src_row"].map(_dxi["dx_label"]) != "Dementia"]
check("[trace] every frozen event source row really carries a Dementia code in the raw table",
      _bad.empty, rid_list(_bad["RID"]))

_cs = _a[_a["event_indicator"] == 0]
_bad = _cs[_cs["censor_src_row"].map(_dxi["DATE"]) != _cs["censor_date"]]
check("[trace] every frozen censor_src_row resolves to the raw DXSUM row with the same censor date",
      _bad.empty, rid_list(_bad["RID"]))
_bad = _cs[~_cs["censor_src_row"].map(_dxi["dx_label"]).isin(["CN", "MCI"])]
check("[trace] every frozen censor source row really carries a CN/MCI code in the raw table",
      _bad.empty, rid_list(_bad["RID"]))

raw DXSUM rows: 16088 | raw plasma rows: 2178
  PASS  [trace] every frozen anchor_src_row resolves to the raw plasma row with the same draw date  [n=0 RIDs=[]]
  PASS  [trace] every frozen anchor_dx_src_row resolves to the raw DXSUM row with the same dx date  [n=0 RIDs=[]]
  PASS  [trace] every frozen event_src_row resolves to the raw DXSUM row with the same event date  [n=0 RIDs=[]]
  PASS  [trace] every frozen event source row really carries a Dementia code in the raw table  [n=0 RIDs=[]]
  PASS  [trace] every frozen censor_src_row resolves to the raw DXSUM row with the same censor date  [n=0 RIDs=[]]
  PASS  [trace] every frozen censor source row really carries a CN/MCI code in the raw table  [n=0 RIDs=[]]


## 3. Re-annotate the source rows  *(annotation only — no re-selection)*

To explain *why* each row was or was not used, we re-apply the **documented** Build 1 rules to the raw
rows: diagnosis harmonization, same-day highest-severity collapse, plasma sentinel/non-positive
cleaning, and the ±90 d nearest-MCI eligibility test.

This produces **annotations**, not a cohort. The consistency assertion below proves the re-annotation
agrees with the frozen anchor for all 535 participants — if it ever disagreed, that would be a bug in
this notebook, not a new cohort.

In [4]:
# ---- diagnosis history: dated + coded, same-day collapse keeping highest severity (Build 1 rule) ----
dx_dated = dx_raw.dropna(subset=["DATE", "dx_code"]).sort_values(["RID", "DATE", "dx_code"])
dx_dated = dx_dated.assign(
    _superseded=dx_dated.duplicated(["RID", "DATE"], keep="last"))   # kept row = highest severity
dxh = dx_dated[~dx_dated["_superseded"]].copy()                      # == Build 1's dxh
SUPERSEDED_ROWS = set(dx_dated.loc[dx_dated["_superseded"], "dx_src_row"])

# ---- plasma cleaning (Build 1 rule) ----
SRC = {"ptau217": "pT217_F", "abeta42": "AB42_F", "abeta40": "AB40_F",
       "nfl": "NfL_Q", "gfap": "GFAP_Q", "ratio_ab42_ab40_vendor": "AB42_AB40_F"}
pl = pl_raw.copy()
for name, col in SRC.items():
    v = pd.to_numeric(pl[col], errors="coerce")
    v = v.mask(v.isin(MISSING_SENTINELS))
    pl[name] = v.mask(v <= 0)
pl["n_usable_core_assays"] = pl[CORE_ASSAYS].notna().sum(axis=1)
pl["_dupe_collapsed"] = pl.dropna(subset=["DATE"]).duplicated(["RID", "DATE"], keep="first") \
                          .reindex(pl.index, fill_value=False)

# ---- nearest dx within +/-90d for EVERY plasma draw -> anchor eligibility ----
_left = pl.dropna(subset=["DATE"]).sort_values("DATE")
_right = (dxh[["RID", "DATE", "dx_label"]].rename(columns={"DATE": "near_dx_date", "dx_label": "near_dx_label"})
          .sort_values("near_dx_date"))
_m = pd.merge_asof(_left, _right, left_on="DATE", right_on="near_dx_date", by="RID",
                   tolerance=pd.Timedelta(days=ALIGN_TOL_DAYS), direction="nearest")
_m["near_dx_offset_days"] = (_m["DATE"] - _m["near_dx_date"]).dt.days
_m["anchor_eligible"] = (_m["n_usable_core_assays"] >= 1) & (_m["near_dx_label"] == "MCI") & (~_m["_dupe_collapsed"])

PL_ANNO = _m.set_index("plasma_src_row")[
    ["near_dx_date", "near_dx_label", "near_dx_offset_days", "anchor_eligible"]]
pl = pl.join(PL_ANNO, on="plasma_src_row")
pl[["anchor_eligible"]] = pl[["anchor_eligible"]].fillna(False)

# ---- CONSISTENCY: the re-annotation reproduces the frozen anchor exactly ----
_recomputed_anchor = (pl[pl["anchor_eligible"]].sort_values(["RID", "DATE", "plasma_src_row"])
                      .drop_duplicates("RID", keep="first")[["RID", "plasma_src_row", "DATE"]]
                      .rename(columns={"plasma_src_row": "recomputed_src_row", "DATE": "recomputed_date"}))
_chk = anchor_frozen[["RID", "anchor_src_row", "anchor_date"]].merge(_recomputed_anchor, on="RID", how="left")
_bad = _chk[(_chk["recomputed_src_row"] != _chk["anchor_src_row"])
            | (_chk["recomputed_date"] != _chk["anchor_date"])]
check("[annotation] re-applying the documented rules reproduces the FROZEN anchor for all "
      f"{N_ANCHOR} participants (annotation is consistent; nothing re-selected)",
      _bad.empty, rid_list(_bad["RID"]))
print(f"\neligible anchor candidate draws: {int(pl['anchor_eligible'].sum())} "
      f"across {pl.loc[pl['anchor_eligible'], 'RID'].nunique()} participants")

  PASS  [annotation] re-applying the documented rules reproduces the FROZEN anchor for all 535 participants (annotation is consistent; nothing re-selected)  [n=0 RIDs=[]]

eligible anchor candidate draws: 656 across 535 participants


## 4. Algorithmic warning flags

Computed for **all 535** anchored participants (so prevalence is reportable), then attached to the
selected cases. **A warning is never an exclusion** — it is a pointer for the human reviewer.

In [5]:
A = anchor_frozen.set_index("RID")
warn = pd.DataFrame(index=A.index)

_dx_by_rid = {r: g for r, g in dxh.sort_values("DATE").groupby("RID")}
_rawdx_by_rid = {r: g for r, g in dx_raw.dropna(subset=["DATE"]).groupby("RID")}
_pl_by_rid = {r: g for r, g in pl.dropna(subset=["DATE"]).groupby("RID")}

rows = []
for rid, r in A.iterrows():
    a = r["anchor_date"]
    g = _dx_by_rid.get(rid, dxh.iloc[0:0])
    graw = _rawdx_by_rid.get(rid, dx_raw.iloc[0:0])
    gpl = _pl_by_rid.get(rid, pl.iloc[0:0])

    pre = g[(g["DATE"] < a)]
    pre_dem = pre[pre["dx_label"] == "Dementia"]
    n_pre_dem = int(len(pre_dem))
    last_pre_dem = pre_dem["DATE"].max() if n_pre_dem else pd.NaT
    days_lastdem_to_anchor = int((a - last_pre_dem).days) if n_pre_dem else np.nan

    # reversion BEFORE the anchor: a non-dementia dx occurring AFTER a pre-anchor dementia dx
    reversion = bool(n_pre_dem and ((pre["DATE"] > last_pre_dem) & (pre["dx_label"].isin(["CN", "MCI"]))).any()
                     or (n_pre_dem and r["anchor_dx_date"] > last_pre_dem))

    ev = (r["event_indicator"] == 1)
    t = r["time_to_event_or_censor_days"]

    # multiple equally-close MCI matches within +/-90d of the anchor
    mci_win = g[(g["dx_label"] == "MCI") & ((g["DATE"] - a).abs() <= pd.Timedelta(days=ALIGN_TOL_DAYS))].copy()
    if len(mci_win):
        d = (mci_win["DATE"] - a).abs()
        n_tied = int((d == d.min()).sum())
    else:
        n_tied = 0

    # conflicting diagnosis codes on the same date (raw, before the highest-severity collapse)
    conflict = False
    if len(graw):
        gg = graw.dropna(subset=["dx_code"])
        conflict = bool(gg.groupby("DATE")["dx_code"].nunique().gt(1).any())

    # non-monotonic (severity decreases at some point)
    nonmono = bool((g.sort_values("DATE")["dx_code"].diff() < 0).any())

    # event AND censor candidate on the same date (raw conflict at the outcome-defining date)
    end = r["followup_end_date"]
    ev_cs_same = False
    if pd.notna(end) and len(graw):
        same = graw[graw["DATE"] == end].dropna(subset=["dx_code"])
        ev_cs_same = bool((same["dx_code"] == 3).any() and (same["dx_code"].isin([1, 2])).any())

    # multiple RAW plasma rows on the anchor date
    n_pl_anchor = int((gpl["DATE"] == a).sum())

    # missing source-row provenance
    miss_prov = bool(pd.isna(r["anchor_src_row"]) or pd.isna(r["anchor_dx_src_row"])
                     or (ev and pd.isna(r["event_src_row"]))
                     or ((r["event_indicator"] == 0) and pd.isna(r["censor_src_row"])))

    offs = [r.get(f"{p}_offset_days") for p in ["cdrsb", "mmse", "moca", "faq"]]
    cog_after = bool(any(pd.notna(o) and o > 0 for o in offs))

    # EXTRA flag (beyond the 15 required): the censoring visit IS the anchor-matching MCI visit,
    # so the participant's ENTIRE follow-up is the anchor->matched-MCI interval (<=90d by construction)
    # and they contribute almost no prospective information.
    censor_is_anchor_dx = bool((r["event_indicator"] == 0)
                               and pd.notna(r["censor_src_row"])
                               and r["censor_src_row"] == r["anchor_dx_src_row"])

    rows.append(dict(
        RID=rid,
        warn_dementia_before_anchor=bool(r["qc_dementia_on_or_before_anchor"]),
        n_pre_anchor_dementia_dx=n_pre_dem,
        days_last_pre_anchor_dementia_to_anchor=days_lastdem_to_anchor,
        warn_reversion_sequence_before_anchor=reversion,
        warn_dementia_within_30d=bool(ev and pd.notna(t) and t <= 30),
        warn_dementia_within_90d=bool(ev and pd.notna(t) and t <= 90),
        warn_dementia_within_180d=bool(ev and pd.notna(t) and t <= RAPID_DAYS),
        warn_anchor_offset_70_90d=bool(BOUNDARY_LO <= abs(r["align_offset_days"]) <= BOUNDARY_HI),
        warn_multiple_equally_close_mci=bool(n_tied >= 2),
        n_equally_close_mci_matches=n_tied,
        warn_multiple_plasma_rows_on_anchor_date=bool(n_pl_anchor > 1),
        n_plasma_rows_on_anchor_date=n_pl_anchor,
        warn_conflicting_dx_same_date=conflict,
        warn_nonmonotonic_dx_sequence=nonmono,
        warn_event_and_censor_same_date=ev_cs_same,
        warn_missing_source_row_provenance=miss_prov,
        warn_cognitive_score_after_anchor=cog_after,     # INFORMATIONAL ONLY
        warn_censor_is_anchor_matching_visit=censor_is_anchor_dx,   # EXTRA (beyond the required 15)
    ))

warn = pd.DataFrame(rows).set_index("RID")

WARN_BOOLS = [c for c in warn.columns if c.startswith("warn_")]
print("Warning-flag prevalence across all 535 anchored participants:")
print(warn[WARN_BOOLS].sum().sort_values(ascending=False).to_string())
check("warning flags computed for every anchored participant", len(warn) == N_ANCHOR)

Warning-flag prevalence across all 535 anchored participants:
warn_cognitive_score_after_anchor           81
warn_censor_is_anchor_matching_visit        74
warn_nonmonotonic_dx_sequence               48
warn_anchor_offset_70_90d                   26
warn_dementia_before_anchor                  8
warn_reversion_sequence_before_anchor        8
warn_dementia_within_180d                    2
warn_multiple_equally_close_mci              1
warn_dementia_within_30d                     0
warn_dementia_within_90d                     0
warn_multiple_plasma_rows_on_anchor_date     0
warn_conflicting_dx_same_date                0
warn_event_and_censor_same_date              0
warn_missing_source_row_provenance           0
  PASS  warning flags computed for every anchored participant


## 5. Deterministic sampling (seed `20260711`)

| # | Stratum | Universe | Rule |
|---|---|---|---|
| **S0** | **mandatory — pre-anchor dementia** | anchor cohort (535) | **all** participants flagged `qc_dementia_on_or_before_anchor` |
| S1 | random — survival cohort | survival cohort (410) | 10 at random |
| S2 | anchor near ±90 d boundary | anchor cohort (535) | 10, **priority** \|offset\| ∈ [70, 90], then next-closest to the boundary |
| S3 | rapid progression | survival-cohort events (86) | 10, **priority** event ≤ 180 d, then next-fastest |
| S4 | random — censored | censored (324) | 10 at random |

Random strata use `np.random.default_rng([20260711, k])` over the **sorted** eligible RID array, so each
stratum is reproducible independently of the others. Ordered strata (S2, S3) use a deterministic sort
with `RID` as the final tie-break — no randomness at all.

Strata are sampled from their **natural universes** and then de-duplicated to unique participants,
**preserving every applicable stratum label**. An under-filled stratum is extended within its *own*
priority ordering — never refilled from a scientifically different stratum. `S2` is drawn from the
anchor cohort (not the survival cohort) because anchor-match quality is a property of the anchor,
including for participants who never reached the survival cohort.

In [6]:
STRATA_DEF = {
    "S0_mandatory_pre_anchor_dementia": "all participants with >=1 dementia dx BEFORE the anchor",
    "S1_random_survival": "10 random participants from the survival cohort",
    "S2_anchor_boundary_70_90d": "10 anchor matches nearest the +/-90d boundary (priority |offset| 70-90d)",
    "S3_rapid_progression": "10 fastest progressors (priority event <=180d)",
    "S4_random_censored": "10 random censored participants",
}


def sample_packet(seed: int) -> tuple[dict, pd.DataFrame]:
    """Deterministic. Returns {stratum: sorted RID array} and the per-stratum summary."""
    fu = followup_frozen
    sel, meta = {}, []

    # ---- S0: MANDATORY, from the Build 1 flag (never hard-coded) ----
    u0 = np.sort(anchor_frozen.loc[anchor_frozen["qc_dementia_on_or_before_anchor"], "RID"].to_numpy())
    sel["S0_mandatory_pre_anchor_dementia"] = u0
    meta.append(dict(stratum_id="S0_mandatory_pre_anchor_dementia", universe="anchor cohort (A)",
                     requested_n=len(u0), eligible_n=len(u0), priority_band_n=len(u0),
                     selected_n=len(u0), shortfall=0,
                     selection_rule="ALL flagged participants (mandatory; no sampling)"))

    # ---- S1: random from the survival cohort ----
    u1 = np.sort(fu["RID"].to_numpy())
    n1 = min(10, len(u1))
    s1 = np.sort(np.random.default_rng([seed, 1]).choice(u1, size=n1, replace=False))
    sel["S1_random_survival"] = s1
    meta.append(dict(stratum_id="S1_random_survival", universe="survival cohort (B)",
                     requested_n=10, eligible_n=len(u1), priority_band_n=len(u1),
                     selected_n=n1, shortfall=10 - n1,
                     selection_rule="rng([seed,1]).choice over sorted RIDs, without replacement"))

    # ---- S2: anchor matches nearest the boundary; priority |offset| in [70,90] ----
    c2 = anchor_frozen[["RID", "align_offset_days_abs"]].copy()
    band2 = int(((c2["align_offset_days_abs"] >= BOUNDARY_LO)
                 & (c2["align_offset_days_abs"] <= BOUNDARY_HI)).sum())
    o2 = c2.sort_values(["align_offset_days_abs", "RID"], ascending=[False, True])
    n2 = min(10, len(o2))
    s2 = np.sort(o2.head(n2)["RID"].to_numpy())
    sel["S2_anchor_boundary_70_90d"] = s2
    meta.append(dict(stratum_id="S2_anchor_boundary_70_90d", universe="anchor cohort (A)",
                     requested_n=10, eligible_n=len(o2), priority_band_n=band2,
                     selected_n=n2, shortfall=max(0, 10 - n2),
                     selection_rule="sort |offset| DESC, RID ASC; take top 10 "
                                    f"(priority band |offset| {BOUNDARY_LO}-{BOUNDARY_HI}d)"))

    # ---- S3: rapid progression; priority event <=180d ----
    c3 = fu[fu["event_indicator"] == 1][["RID", "time_to_event_or_censor_days"]].copy()
    band3 = int((c3["time_to_event_or_censor_days"] <= RAPID_DAYS).sum())
    o3 = c3.sort_values(["time_to_event_or_censor_days", "RID"], ascending=[True, True])
    n3 = min(10, len(o3))
    s3 = np.sort(o3.head(n3)["RID"].to_numpy())
    sel["S3_rapid_progression"] = s3
    meta.append(dict(stratum_id="S3_rapid_progression", universe="survival-cohort events",
                     requested_n=10, eligible_n=len(o3), priority_band_n=band3,
                     selected_n=n3, shortfall=max(0, 10 - n3),
                     selection_rule=f"sort time-to-event ASC, RID ASC; take top 10 "
                                    f"(priority band <= {RAPID_DAYS}d)"))

    # ---- S4: random censored ----
    u4 = np.sort(fu.loc[fu["event_indicator"] == 0, "RID"].to_numpy())
    n4 = min(10, len(u4))
    s4 = np.sort(np.random.default_rng([seed, 4]).choice(u4, size=n4, replace=False))
    sel["S4_random_censored"] = s4
    meta.append(dict(stratum_id="S4_random_censored", universe="censored participants",
                     requested_n=10, eligible_n=len(u4), priority_band_n=len(u4),
                     selected_n=n4, shortfall=10 - n4,
                     selection_rule="rng([seed,4]).choice over sorted RIDs, without replacement"))

    return sel, pd.DataFrame(meta)


selection, strata_meta = sample_packet(RANDOM_SEED)

# ---- ASSERTION 10: sampling is reproducible ----
selection2, _ = sample_packet(RANDOM_SEED)
check("sampling is reproducible under seed 20260711 (identical selection on a second call)",
      all(np.array_equal(selection[k], selection2[k]) for k in selection))

for k, v in selection.items():
    print(f"  {k:36s} n={len(v):>2d}  {list(v)}")

# ---- de-duplicate, preserving EVERY applicable stratum label ----
label_map: dict[int, list[str]] = {}
for k, v in selection.items():
    for r in v:
        label_map.setdefault(int(r), []).append(k)

SELECTED = np.sort(np.array(list(label_map.keys())))
n_before_dedup = int(sum(len(v) for v in selection.values()))
overlap_rids = sorted(r for r, ls in label_map.items() if len(ls) > 1)

print(f"\nselected before de-duplication : {n_before_dedup}")
print(f"unique participants after dedup: {len(SELECTED)}")
print(f"participants in >1 stratum     : {len(overlap_rids)} -> {overlap_rids}")
for r in overlap_rids:
    print(f"    RID {r}: {' | '.join(sorted(label_map[r]))}")

  PASS  sampling is reproducible under seed 20260711 (identical selection on a second call)
  S0_mandatory_pre_anchor_dementia     n= 8  [np.int64(467), np.int64(4115), np.int64(4506), np.int64(4706), np.int64(6976), np.int64(7070), np.int64(10251), np.int64(10322)]
  S1_random_survival                   n=10  [np.int64(448), np.int64(1222), np.int64(2180), np.int64(6007), np.int64(6312), np.int64(6362), np.int64(6605), np.int64(6889), np.int64(6947), np.int64(7016)]
  S2_anchor_boundary_70_90d            n=10  [np.int64(2200), np.int64(2392), np.int64(4067), np.int64(4742), np.int64(4809), np.int64(6696), np.int64(6883), np.int64(10098), np.int64(10843), np.int64(10850)]
  S3_rapid_progression                 n=10  [np.int64(256), np.int64(269), np.int64(567), np.int64(723), np.int64(4240), np.int64(4366), np.int64(4796), np.int64(6345), np.int64(6939), np.int64(7088)]
  S4_random_censored                   n=10  [np.int64(2180), np.int64(4351), np.int64(6240), np.int64(6696), np.int6

## 6. QC cases table — one row per unique selected participant

In [7]:
cases = anchor_frozen[anchor_frozen["RID"].isin(SELECTED)].copy()
cases["strata"] = cases["RID"].map(lambda r: "|".join(sorted(label_map[int(r)])))
cases["n_strata"] = cases["RID"].map(lambda r: len(label_map[int(r)]))
cases["is_mandatory_pre_anchor_dementia"] = cases["RID"].isin(MANDATORY)
cases = cases.merge(warn.reset_index(), on="RID", how="left")

CASE_COLS = [
    "RID", "PTID", "strata", "n_strata", "is_mandatory_pre_anchor_dementia",
    # anchor
    "anchor_date", "anchor_viscode", "anchor_dx_date", "anchor_dx_viscode",
    "align_offset_days", "align_offset_days_abs", "anchor_match_type",
    # outcome
    "event_indicator", "event_date", "censor_date", "followup_end_date",
    "time_to_event_or_censor_days", "time_to_event_or_censor_months", "n_post_anchor_visits",
    "censoring_reason", "reverted_cn",
    # cohort membership
    "followup_eligible", "primary_eligible", "final_inclusion_stage",
    "exclusion_stage", "exclusion_reason",
    # source-row provenance
    "anchor_src_row", "anchor_dx_src_row", "anchor_dx_src_id",
    "event_src_row", "event_src_id", "censor_src_row", "censor_src_id",
    # warning flags
    "warn_dementia_before_anchor", "n_pre_anchor_dementia_dx",
    "days_last_pre_anchor_dementia_to_anchor", "warn_reversion_sequence_before_anchor",
    "warn_dementia_within_30d", "warn_dementia_within_90d", "warn_dementia_within_180d",
    "warn_anchor_offset_70_90d", "warn_multiple_equally_close_mci", "n_equally_close_mci_matches",
    "warn_multiple_plasma_rows_on_anchor_date", "n_plasma_rows_on_anchor_date",
    "warn_conflicting_dx_same_date", "warn_nonmonotonic_dx_sequence",
    "warn_event_and_censor_same_date", "warn_missing_source_row_provenance",
    "warn_cognitive_score_after_anchor", "warn_censor_is_anchor_matching_visit",
    # baseline values the reviewer may want in view (read-only context)
    "entry_age", "APOE4_COUNT", "ptau217", "n_usable_core_assays",
]
qc_cases = cases[CASE_COLS].sort_values("RID").reset_index(drop=True)

# a compact human-readable warning summary for the form
def warn_summary(r) -> str:
    hits = [c.replace("warn_", "") for c in WARN_BOOLS if bool(r[c])]
    return "|".join(hits) if hits else "none"


qc_cases["warning_flags_summary"] = qc_cases.apply(warn_summary, axis=1)
print(f"QC cases: {qc_cases.shape[0]} participants x {qc_cases.shape[1]} columns")
print(qc_cases[["RID", "strata", "is_mandatory_pre_anchor_dementia", "align_offset_days",
                "event_indicator", "time_to_event_or_censor_days",
                "primary_eligible", "warning_flags_summary"]].to_string(index=False))

QC cases: 46 participants x 56 columns
  RID                                       strata  is_mandatory_pre_anchor_dementia  align_offset_days  event_indicator  time_to_event_or_censor_days  primary_eligible                                                                                            warning_flags_summary
  256                         S3_rapid_progression                             False               15.0              1.0                         167.0              True                                                                                             dementia_within_180d
  269                         S3_rapid_progression                             False                0.0              1.0                         188.0              True                                                                                                             none
  448                           S1_random_survival                             False                0.0            

## 7. Longitudinal context (long format)

**Full diagnosis history and all plasma draws** for every selected participant — not a narrow window.
This exceeds the "bounded window" allowance for ordinary participants and is what the eight mandatory
cases require. Full history is what lets a reviewer independently confirm that

- the anchor is the **earliest** eligible draw (every competing draw is shown, with the reason it was
  or was not eligible);
- the event is the **first** post-anchor dementia (every earlier post-anchor diagnosis is shown);
- the censor is the **last** qualifying post-anchor non-dementia visit (every later row is shown, with
  the reason it does not qualify).

Undated rows and unknown/other-coded rows are **included** and annotated, because their exclusion from
the outcome logic is itself something the reviewer must be able to see.

In [8]:
ctx_rows = []
for rid in SELECTED:
    r = A.loc[rid]
    a = r["anchor_date"]
    ev_row = r["event_src_row"]
    cs_row = r["censor_src_row"]
    mci_row = r["anchor_dx_src_row"]
    an_row = r["anchor_src_row"]

    g = dx_raw[dx_raw["RID"] == rid]
    gd = g.dropna(subset=["DATE", "dx_code"])
    gd = gd[~gd["dx_src_row"].isin(SUPERSEDED_ROWS)]
    post = gd[gd["DATE"] > a]
    post_dem = post[post["dx_label"] == "Dementia"].sort_values("DATE")
    post_nd = post[post["dx_label"].isin(["CN", "MCI"])].sort_values("DATE")
    first_dem = post_dem["DATE"].iloc[0] if len(post_dem) else pd.NaT
    last_nd = post_nd["DATE"].iloc[-1] if len(post_nd) else pd.NaT

    # alternative MCI matches within +/-90d of the anchor
    alt_mci = set(gd.loc[(gd["dx_label"] == "MCI")
                         & ((gd["DATE"] - a).abs() <= pd.Timedelta(days=ALIGN_TOL_DAYS))
                         & (gd["dx_src_row"] != mci_row), "dx_src_row"])

    # ---------- diagnosis rows ----------
    for _, d in g.iterrows():
        roles, alt = [], False
        srow = d["dx_src_row"]
        dt = d["DATE"]
        if srow == mci_row:
            roles.append("MATCHED_MCI")
        if pd.notna(ev_row) and srow == ev_row:
            roles.append("EVENT")
        if pd.notna(cs_row) and srow == cs_row:
            roles.append("CENSOR")

        if pd.isna(dt):
            pos, reason = "undated", "NOT USED: missing EXAMDATE -> excluded from all date logic"
        else:
            pos = "pre_anchor" if dt < a else ("at_anchor" if dt == a else "post_anchor")
            if pd.isna(d["dx_code"]):
                reason = "NOT USED: unknown/other diagnosis code -> never defines an event or a censor"
            elif srow in SUPERSEDED_ROWS:
                reason = "NOT USED: same-date record superseded by the higher-severity code on this date"
            elif dt < a:
                reason = "pre-anchor context"
                if d["dx_label"] == "Dementia":
                    reason = ">>> DEMENTIA BEFORE ANCHOR — prevalent-dementia adjudication required <<<"
            elif dt == a:
                reason = "at-anchor context (a same-day dementia is never a prospective event)"
            elif d["dx_label"] == "Dementia":
                reason = ("SELECTED: first post-anchor dementia -> EVENT" if dt == first_dem
                          else "not used: a later dementia (the event is the FIRST post-anchor dementia)")
            else:
                if pd.notna(first_dem) and dt < first_dem:
                    reason = "post-anchor non-dementia BEFORE the event (context: confirms the event is first)"
                elif pd.notna(first_dem):
                    reason = "not used: non-dementia AFTER the event (follow-up ends at the event)"
                elif dt == last_nd:
                    reason = "SELECTED: last post-anchor non-dementia -> CENSOR"
                else:
                    reason = "not used: not the last post-anchor non-dementia visit"
            if srow == mci_row:
                reason = "SELECTED: matched MCI diagnosis for the anchor (+/-90d nearest) | " + reason
            if srow in alt_mci:
                alt = True
                reason = "ALTERNATIVE eligible MCI match within +/-90d (not nearest / not selected) | " + reason

        ctx_rows.append(dict(
            RID=rid, PTID=d["PTID"], context_scope="full_history",
            source_table="DXSUM", source_row_id=int(srow),
            source_native_id=d["dx_src_id"], record_type="diagnosis",
            record_date=dt, viscode=d["VISCODE2"],
            days_from_anchor=(int((dt - a).days) if pd.notna(dt) else np.nan),
            temporal_position=pos,
            row_roles="|".join(roles), is_selected=bool(roles), is_alternative_candidate=alt,
            selection_reason=reason,
            dx_code_raw=d["DIAGNOSIS"], dx_label=d["dx_label"],
            dx_confidence=d["DXCONFID"], dx_norm=d["DXNORM"], dx_mci=d["DXMCI"], dx_ad=d["DXAD"],
            raw_EXAMDATE=d["raw_EXAMDATE"],
            ptau217=np.nan, abeta42=np.nan, abeta40=np.nan, nfl=np.nan, gfap=np.nan,
            n_usable_core_assays=np.nan, assay_usability="", plasma_anchor_eligible="",
            plasma_nearest_dx_date=pd.NaT, plasma_nearest_dx_label="", plasma_dx_offset_days=np.nan,
            raw_pT217_F="", raw_AB42_F="", raw_AB40_F="", raw_NfL_Q="", raw_GFAP_Q="",
            raw_AB42_AB40_F="", plasma_comment="",
        ))

    # ---------- plasma rows ----------
    for _, p in pl[pl["RID"] == rid].iterrows():
        srow = p["plasma_src_row"]
        dt = p["DATE"]
        is_anchor = (srow == an_row)
        elig = bool(p["anchor_eligible"])
        roles = ["ANCHOR"] if is_anchor else []

        if pd.isna(dt):
            pos, reason = "undated", "NOT ELIGIBLE: missing draw date"
        else:
            pos = "pre_anchor" if dt < a else ("at_anchor" if dt == a else "post_anchor")
            if is_anchor:
                reason = ("SELECTED ANCHOR: earliest eligible draw "
                          f"({int(p['n_usable_core_assays'])}/5 usable core assays; nearest dx = "
                          f"{p['near_dx_label']} at {p['near_dx_offset_days']:+.0f}d)")
            elif elig:
                reason = ("ALTERNATIVE eligible draw — NOT selected because it is LATER than the anchor "
                          "(the earliest eligible draw always wins; completeness is never preferred)"
                          if dt > a else
                          "ALTERNATIVE eligible draw EARLIER than the anchor — INVESTIGATE (should not occur)")
            elif p["_dupe_collapsed"]:
                reason = "NOT ELIGIBLE: duplicate (RID, draw-date) row collapsed (keep=first)"
            elif p["n_usable_core_assays"] < 1:
                reason = "NOT ELIGIBLE: no usable core assay (all of p-tau217/Ab42/Ab40/NfL/GFAP missing)"
            elif pd.isna(p["near_dx_label"]):
                reason = f"NOT ELIGIBLE: no diagnosis within +/-{ALIGN_TOL_DAYS}d of this draw"
            else:
                reason = (f"NOT ELIGIBLE: nearest diagnosis within +/-{ALIGN_TOL_DAYS}d is "
                          f"{p['near_dx_label']} ({p['near_dx_offset_days']:+.0f}d), not MCI")

        ctx_rows.append(dict(
            RID=rid, PTID=p["PTID"], context_scope="full_history",
            source_table="UPENN_PLASMA", source_row_id=int(srow),
            source_native_id="", record_type="plasma",
            record_date=dt, viscode=p["VISCODE2"],
            days_from_anchor=(int((dt - a).days) if pd.notna(dt) else np.nan),
            temporal_position=pos,
            row_roles="|".join(roles), is_selected=bool(roles),
            is_alternative_candidate=bool(elig and not is_anchor),
            selection_reason=reason,
            dx_code_raw="", dx_label="", dx_confidence="", dx_norm="", dx_mci="", dx_ad="",
            raw_EXAMDATE=p["raw_EXAMDATE"],
            ptau217=p["ptau217"], abeta42=p["abeta42"], abeta40=p["abeta40"],
            nfl=p["nfl"], gfap=p["gfap"],
            n_usable_core_assays=p["n_usable_core_assays"],
            assay_usability=f"{int(p['n_usable_core_assays'])}/5 core assays usable",
            plasma_anchor_eligible=elig,
            plasma_nearest_dx_date=p["near_dx_date"], plasma_nearest_dx_label=p["near_dx_label"],
            plasma_dx_offset_days=p["near_dx_offset_days"],
            raw_pT217_F=p["pT217_F"], raw_AB42_F=p["AB42_F"], raw_AB40_F=p["AB40_F"],
            raw_NfL_Q=p["NfL_Q"], raw_GFAP_Q=p["GFAP_Q"], raw_AB42_AB40_F=p["AB42_AB40_F"],
            plasma_comment=p["Comment"],
        ))

qc_context = pd.DataFrame(ctx_rows)
# deterministic sort: RID, date (undated last), source table, source row id
qc_context["_nulldate"] = qc_context["record_date"].isna().astype(int)
qc_context = (qc_context.sort_values(["RID", "_nulldate", "record_date", "source_table", "source_row_id"])
              .drop(columns=["_nulldate"]).reset_index(drop=True))

print(f"Longitudinal context: {qc_context.shape[0]} rows x {qc_context.shape[1]} cols "
      f"for {qc_context['RID'].nunique()} participants")
print(qc_context.groupby("record_type").size().to_string())
print("\nselected rows by role:")
print(qc_context.loc[qc_context["is_selected"], "row_roles"].value_counts().to_string())

Longitudinal context: 362 rows x 40 cols for 46 participants
record_type
diagnosis    288
plasma        74

selected rows by role:
row_roles
ANCHOR                46
MATCHED_MCI           37
CENSOR                19
EVENT                 13
MATCHED_MCI|CENSOR     9


## 8. QC form — **all human-review fields blank**

The form is the reviewer's worksheet. Every `qc_*`, `reviewer_*`, `review_*` and `recommended_action`
field is written **blank** and is asserted blank. Nothing is auto-adjudicated.

### Allowed values

**General status** (`qc_anchor_status`, `qc_outcome_status`, `qc_overall_status`,
`qc_prior_dementia_history_status`, `qc_diagnostic_reversion_interpretation`,
`qc_sensitivity_analysis_recommendation`)
`PASS` · `FAIL` · `UNCERTAIN` · `NOT_REVIEWED`

**`qc_prior_dementia_record_plausibility`**
`PLAUSIBLE_TRUE_DEMENTIA` · `LIKELY_CODING_OR_DATA_ERROR` · `INSUFFICIENT_INFORMATION` · `NOT_APPLICABLE`

**`qc_incident_event_validity`**
`PLAUSIBLE_INCIDENT_PROGRESSION` · `LIKELY_REDIAGNOSIS_OF_PREVALENT_DEMENTIA` ·
`POSSIBLE_REVERSION_THEN_PROGRESSION` · `INSUFFICIENT_INFORMATION` · `NOT_APPLICABLE`

**`qc_primary_cohort_recommendation`**
`RETAIN_PRIMARY` · `EXCLUDE_PRIMARY_RETAIN_SENSITIVITY` · `EXCLUDE_ALL_ANALYSES` ·
`REQUIRES_TEAM_ADJUDICATION` · `NOT_APPLICABLE`

In [9]:
HUMAN_FIELDS = [
    "qc_anchor_status", "qc_outcome_status", "qc_overall_status",
    "qc_prior_dementia_history_status", "qc_prior_dementia_record_plausibility",
    "qc_incident_event_validity", "qc_diagnostic_reversion_interpretation",
    "qc_primary_cohort_recommendation", "qc_sensitivity_analysis_recommendation",
    "recommended_action", "reviewer_name", "review_date", "reviewer_notes",
]

qc_form = qc_cases[[
    "RID", "PTID", "strata", "is_mandatory_pre_anchor_dementia",
    "anchor_date", "anchor_dx_date", "align_offset_days", "align_offset_days_abs",
    "event_indicator", "event_date", "censor_date", "time_to_event_or_censor_days",
    "followup_eligible", "primary_eligible",
    "n_pre_anchor_dementia_dx", "days_last_pre_anchor_dementia_to_anchor",
    "warning_flags_summary",
]].copy()

qc_form = qc_form.rename(columns={"anchor_dx_date": "matched_mci_date",
                                  "align_offset_days": "day_difference_signed",
                                  "align_offset_days_abs": "day_difference_abs",
                                  "event_indicator": "event_status",
                                  "time_to_event_or_censor_days": "followup_days"})

for c in HUMAN_FIELDS:
    qc_form[c] = ""                      # BLANK — the reviewer fills these in

qc_form = qc_form.sort_values("RID").reset_index(drop=True)

check("all human-review fields are blank in the QC form",
      bool((qc_form[HUMAN_FIELDS] == "").all().all()),
      f"{len(HUMAN_FIELDS)} fields x {len(qc_form)} rows")
check("every selected participant appears exactly once in the QC form",
      len(qc_form) == len(SELECTED) and not qc_form["RID"].duplicated().any()
      and set(qc_form["RID"]) == set(SELECTED))
print(f"QC form: {qc_form.shape[0]} rows x {qc_form.shape[1]} cols "
      f"({len(HUMAN_FIELDS)} blank human-review fields)")

  PASS  all human-review fields are blank in the QC form  [13 fields x 46 rows]
  PASS  every selected participant appears exactly once in the QC form
QC form: 46 rows x 30 cols (13 blank human-review fields)


## 9. Sampling summary (requested / eligible / selected / dedup / overlap)

In [10]:
rowsum = []
for _, r in strata_meta.iterrows():
    sid = r["stratum_id"]
    rowsum.append(dict(row_type="stratum", stratum_id=sid, description=STRATA_DEF[sid],
                       universe=r["universe"], selection_rule=r["selection_rule"],
                       requested_n=r["requested_n"], eligible_n=r["eligible_n"],
                       priority_band_n=r["priority_band_n"], selected_n=r["selected_n"],
                       shortfall=r["shortfall"],
                       rids="|".join(str(int(x)) for x in selection[sid])))

rowsum.append(dict(row_type="summary", stratum_id="TOTAL_BEFORE_DEDUP", description="sum of all strata",
                   universe="", selection_rule="", requested_n=int(strata_meta["requested_n"].sum()),
                   eligible_n="", priority_band_n="", selected_n=n_before_dedup, shortfall="", rids=""))
rowsum.append(dict(row_type="summary", stratum_id="TOTAL_UNIQUE_AFTER_DEDUP",
                   description="unique participants in the packet", universe="", selection_rule="",
                   requested_n="", eligible_n="", priority_band_n="", selected_n=len(SELECTED),
                   shortfall="", rids="|".join(str(int(x)) for x in SELECTED)))
rowsum.append(dict(row_type="summary", stratum_id="OVERLAP_MULTI_STRATUM",
                   description="participants appearing in >1 stratum", universe="", selection_rule="",
                   requested_n="", eligible_n="", priority_band_n="", selected_n=len(overlap_rids),
                   shortfall="", rids="|".join(str(int(x)) for x in overlap_rids)))

keys = list(selection)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        inter = sorted(set(selection[keys[i]].tolist()) & set(selection[keys[j]].tolist()))
        if inter:
            rowsum.append(dict(row_type="overlap_pair", stratum_id=f"{keys[i]} & {keys[j]}",
                               description="participants shared by these two strata", universe="",
                               selection_rule="", requested_n="", eligible_n="", priority_band_n="",
                               selected_n=len(inter), shortfall="",
                               rids="|".join(str(int(x)) for x in inter)))

qc_sampling = pd.DataFrame(rowsum)
print(qc_sampling[["row_type", "stratum_id", "requested_n", "eligible_n",
                   "priority_band_n", "selected_n", "shortfall"]].to_string(index=False))

    row_type                                     stratum_id requested_n eligible_n priority_band_n  selected_n shortfall
     stratum               S0_mandatory_pre_anchor_dementia           8          8               8           8         0
     stratum                             S1_random_survival          10        410             410          10         0
     stratum                      S2_anchor_boundary_70_90d          10        535              26          10         0
     stratum                           S3_rapid_progression          10         86               2          10         0
     stratum                             S4_random_censored          10        324             324          10         0
     summary                             TOTAL_BEFORE_DEDUP          48                                     48          
     summary                       TOTAL_UNIQUE_AFTER_DEDUP                                                 46          
     summary                    

## 10. Validation

In [11]:
# ---- 1. all 8 mandatory participants are in the packet ----
missing_mand = set(MANDATORY) - set(SELECTED)
check("ALL pre-anchor-dementia participants are included in the QC packet (mandatory stratum)",
      not missing_mand, f"mandatory n={len(MANDATORY)}; missing={rid_list(missing_mand)}")
check("mandatory stratum was derived from the Build 1 flag, not a hard-coded list",
      set(MANDATORY) == set(anchor_frozen.loc[anchor_frozen["qc_dementia_on_or_before_anchor"], "RID"]))

# ---- 3. all applicable strata preserved after dedup ----
for sid, rids in selection.items():
    lab = qc_cases.set_index("RID")["strata"]
    bad = [int(r) for r in rids if sid not in lab.loc[int(r)].split("|")]
    check(f"[strata] every sampled participant of {sid} retains that label after de-duplication",
          not bad, rid_list(bad))
check("every packet participant carries >=1 stratum label",
      bool((qc_cases["n_strata"] >= 1).all()))

# ---- 5. traceability of every selected row (already round-tripped in cell 3 for all 535) ----
sel_ctx = qc_context[qc_context["is_selected"]]
for role in ["ANCHOR", "MATCHED_MCI", "EVENT", "CENSOR"]:
    have = set(sel_ctx.loc[sel_ctx["row_roles"].str.contains(role), "RID"])
    if role == "ANCHOR":
        exp = set(SELECTED)
    elif role == "MATCHED_MCI":
        exp = set(SELECTED)
    elif role == "EVENT":
        exp = set(qc_cases.loc[qc_cases["event_indicator"] == 1, "RID"])
    else:
        exp = set(qc_cases.loc[qc_cases["event_indicator"] == 0, "RID"])
    check(f"[trace] every packet participant with a {role} row has it present and role-tagged in context",
          have == exp, f"expected {len(exp)}, found {len(have)}; missing={rid_list(exp - have)}")

check("[trace] every context row carries a source table + source row id",
      bool(qc_context["source_row_id"].notna().all() and qc_context["source_table"].notna().all()))

# ---- 6. each event is shown with ALL earlier post-anchor diagnoses ----
bad_ev = []
for rid in qc_cases.loc[qc_cases["event_indicator"] == 1, "RID"]:
    a = A.loc[rid, "anchor_date"]
    ed = A.loc[rid, "event_date"]
    expect = set(dxh.loc[(dxh["RID"] == rid) & (dxh["DATE"] > a) & (dxh["DATE"] <= ed), "dx_src_row"])
    got = set(qc_context.loc[qc_context["RID"] == rid, "source_row_id"])
    if not expect <= got:
        bad_ev.append(int(rid))
check("every EVENT is shown in context together with ALL earlier post-anchor diagnoses "
      "(so the reviewer can confirm it is the FIRST)", not bad_ev, rid_list(bad_ev))

# ---- 7. each censor shown with evidence that no later qualifying diagnosis exists ----
bad_cs = []
for rid in qc_cases.loc[qc_cases["event_indicator"] == 0, "RID"]:
    a = A.loc[rid, "anchor_date"]
    cd = A.loc[rid, "censor_date"]
    expect = set(dxh.loc[(dxh["RID"] == rid) & (dxh["DATE"] > a), "dx_src_row"])   # ALL post-anchor
    got = set(qc_context.loc[qc_context["RID"] == rid, "source_row_id"])
    later_qualifying = dxh[(dxh["RID"] == rid) & (dxh["DATE"] > cd)]
    if not expect <= got or len(later_qualifying):
        bad_cs.append(int(rid))
check("every CENSOR is shown with ALL post-anchor diagnoses and no later qualifying diagnosis exists",
      not bad_cs, rid_list(bad_cs))

# ---- 8. the packet does not modify Build 1 inclusion ----
check("QC packet does not change Build 1 participant inclusion (cohort sizes unchanged)",
      (len(anchor_frozen), len(followup_frozen), len(primary_frozen)) == (N_ANCHOR, N_FU, N_PRIMARY)
      and set(MANDATORY) <= set(anchor_frozen["RID"]))
check("all 8 pre-anchor-dementia participants REMAIN in the frozen v1 cohort (none excluded here)",
      set(MANDATORY) <= set(anchor_frozen["RID"])
      and int(anchor_frozen.loc[anchor_frozen["RID"].isin(MANDATORY), "followup_eligible"].sum())
          == int(followup_frozen["RID"].isin(MANDATORY).sum()))

  PASS  ALL pre-anchor-dementia participants are included in the QC packet (mandatory stratum)  [mandatory n=8; missing=n=0 RIDs=[]]
  PASS  mandatory stratum was derived from the Build 1 flag, not a hard-coded list
  PASS  [strata] every sampled participant of S0_mandatory_pre_anchor_dementia retains that label after de-duplication  [n=0 RIDs=[]]
  PASS  [strata] every sampled participant of S1_random_survival retains that label after de-duplication  [n=0 RIDs=[]]
  PASS  [strata] every sampled participant of S2_anchor_boundary_70_90d retains that label after de-duplication  [n=0 RIDs=[]]
  PASS  [strata] every sampled participant of S3_rapid_progression retains that label after de-duplication  [n=0 RIDs=[]]
  PASS  [strata] every sampled participant of S4_random_censored retains that label after de-duplication  [n=0 RIDs=[]]
  PASS  every packet participant carries >=1 stratum label
  PASS  [trace] every packet participant with a ANCHOR row has it present and role-tagged in context  

## 11. Pre-anchor-dementia adjudication packet — the eight mandatory cases

The specific validation the spec requires: how many reach the survival cohort, how many reach the
primary cohort, how many are counted as post-anchor **events**, and **which primary-cohort events are
affected**. No exclusion decision is made in code.

In [12]:
mand = anchor_frozen[anchor_frozen["RID"].isin(MANDATORY)].merge(warn.reset_index(), on="RID")
m_fu = mand[mand["followup_eligible"]]
m_pr = mand[mand["primary_eligible"]]
m_ev = mand[mand["event_indicator"] == 1]
m_pr_ev = mand[(mand["primary_eligible"]) & (mand["event_indicator"] == 1)]

print("=" * 100)
print("PRE-ANCHOR DEMENTIA — MANDATORY ADJUDICATION CASES")
print("=" * 100)
print(mand[["RID", "anchor_date", "anchor_dx_date", "align_offset_days",
            "n_pre_anchor_dementia_dx", "days_last_pre_anchor_dementia_to_anchor",
            "warn_reversion_sequence_before_anchor",
            "followup_eligible", "primary_eligible", "event_indicator", "event_date",
            "censor_date", "time_to_event_or_censor_days"]].to_string(index=False))

print(f"\nmandatory cases identified/included : {len(MANDATORY)} / {len(mand)}")
print(f"reach the survival cohort (B)       : {len(m_fu)}   (events among them: {len(m_ev)})")
print(f"reach the primary cohort (C)        : {len(m_pr)}   (events among them: {len(m_pr_ev)})")
print(f"AFFECTED PRIMARY-COHORT EVENT(S)    : {sorted(int(r) for r in m_pr_ev['RID'])}")
print("\nIf the team later excluded all 8 (COMPUTED, NOT APPLIED):")
print(f"  A {N_ANCHOR} -> {N_ANCHOR - len(mand)} | B {N_FU} -> {N_FU - len(m_fu)} "
      f"(events {N_FU_EVENTS} -> {N_FU_EVENTS - len(m_ev)}) | "
      f"C {N_PRIMARY} -> {N_PRIMARY - len(m_pr)} (events {N_PRIMARY_EVENTS} -> {N_PRIMARY_EVENTS - len(m_pr_ev)})")

print("\n" + "=" * 100)
print("DIAGNOSIS TIMELINES (dated, coded records; '*' marks a selected row)")
print("=" * 100)
for rid in MANDATORY:
    r = A.loc[rid]
    a = r["anchor_date"]
    g = dxh[dxh["RID"] == rid].sort_values("DATE")
    print(f"\nRID {rid}  anchor={a.date()}  event={r['event_date']}  censor={r['censor_date']}  "
          f"in_B={bool(r['followup_eligible'])}  in_C={bool(r['primary_eligible'])}")
    for _, d in g.iterrows():
        rel = (d["DATE"] - a).days
        mk = []
        if d["dx_src_row"] == r["anchor_dx_src_row"]:
            mk.append("MATCHED_MCI")
        if pd.notna(r["event_src_row"]) and d["dx_src_row"] == r["event_src_row"]:
            mk.append("EVENT")
        if pd.notna(r["censor_src_row"]) and d["dx_src_row"] == r["censor_src_row"]:
            mk.append("CENSOR")
        star = "*" if mk else " "
        print(f"   {star} {d['DATE'].date()}  {rel:+6d}d  {d['dx_label']:<14s} "
              f"viscode={str(d['VISCODE2']):<6s} conf={d['DXCONFID']} row={int(d['dx_src_row'])} "
              f"{'  <- ' + '+'.join(mk) if mk else ''}")

n_ctx_mand = int(qc_context["RID"].isin(MANDATORY).sum())
print(f"\ncontext rows produced for the 8 mandatory cases: {n_ctx_mand}")

PRE-ANCHOR DEMENTIA — MANDATORY ADJUDICATION CASES
  RID anchor_date anchor_dx_date  align_offset_days  n_pre_anchor_dementia_dx  days_last_pre_anchor_dementia_to_anchor  warn_reversion_sequence_before_anchor  followup_eligible  primary_eligible  event_indicator event_date censor_date  time_to_event_or_censor_days
  467  2017-09-13     2017-09-19               -6.0                         5                                    450.0                                   True               True              True              1.0 2018-09-21         NaT                         373.0
 4115  2024-01-24     2024-01-19                5.0                         4                                    726.0                                   True               True              True              0.0        NaT  2025-02-05                         378.0
 4506  2018-11-15     2018-12-11              -26.0                         2                                   1331.0                                   T

## 12. Write the QC packet + verify Build 1 is byte-identical

In [13]:
DATE_OUT = ["anchor_date", "anchor_dx_date", "matched_mci_date", "event_date", "censor_date",
            "followup_end_date", "record_date", "plasma_nearest_dx_date"]


def datestr(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for c in DATE_OUT:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors="coerce").dt.strftime("%Y-%m-%d")
    return d


OUTPUTS = {
    f"mci_survival_manual_qc_cases_{VERSION}.tsv": datestr(qc_cases),
    f"mci_survival_manual_qc_longitudinal_context_{VERSION}.tsv": datestr(qc_context),
    f"mci_survival_manual_qc_form_{VERSION}.tsv": datestr(qc_form),
    f"mci_survival_manual_qc_sampling_summary_{VERSION}.tsv": qc_sampling,
}
for name in OUTPUTS:
    if (FREEZE_DIR / name).exists():
        print(f"  NOTE: overwriting this build's own prior output: {name}")

paths = {n: save_tsv(d, n) for n, d in OUTPUTS.items()}

# ---- 9. Build 1 files byte-identical BEFORE vs AFTER ----
HASHES_AFTER = {f: sha256(FREEZE_DIR / f) for f in BUILD1_FILES}
drift = [f for f in BUILD1_FILES if HASHES_BEFORE[f] != HASHES_AFTER[f]]
check("Build 1 outputs are BYTE-IDENTICAL before and after Build 2 (nothing overwritten)",
      not drift, f"drifted: {drift}" if drift else "all 6 Build 1 artifacts unchanged")

print("\nBuild 1 freeze — hashes AFTER Build 2 (unchanged):")
for f in BUILD1_FILES:
    print(f"  {f:44s} {HASHES_AFTER[f][:16]}...  {'OK' if HASHES_BEFORE[f] == HASHES_AFTER[f] else 'CHANGED!'}")

print("\nQC packet hashes:")
for n, p in paths.items():
    print(f"  {n:52s} {sha256(p)[:16]}...")

  NOTE: overwriting this build's own prior output: mci_survival_manual_qc_cases_v1.tsv


  NOTE: overwriting this build's own prior output: mci_survival_manual_qc_longitudinal_context_v1.tsv
  NOTE: overwriting this build's own prior output: mci_survival_manual_qc_form_v1.tsv
  NOTE: overwriting this build's own prior output: mci_survival_manual_qc_sampling_summary_v1.tsv
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_manual_qc_cases_v1.tsv   (46 x 56)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_manual_qc_longitudinal_context_v1.tsv   (362 x 40)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_manual_qc_form_v1.tsv   (46 x 30)


  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_manual_qc_sampling_summary_v1.tsv   (10 x 11)
  PASS  Build 1 outputs are BYTE-IDENTICAL before and after Build 2 (nothing overwritten)  [all 6 Build 1 artifacts unchanged]

Build 1 freeze — hashes AFTER Build 2 (unchanged):
  mci_survival_anchor_cohort_v1.tsv            64e66814726c2263...  OK
  mci_survival_followup_cohort_v1.tsv          8203a70f7cc8d030...  OK
  mci_survival_primary_cohort_v1.tsv           a15e2cb8606f676b...  OK
  mci_survival_exclusion_log_v1.tsv            a3eafebcdd3e392b...  OK
  mci_survival_cohort_flow_v1.tsv              39b88d1a9f6cc3e4...  OK
  mci_survival_freeze_provenance_v1.json       d32c113a0406242b...  OK

QC packet hashes:
  mci_survival_manual_qc_cases_v1.tsv                  90669ba268a39dae...
  mci_survival_manual_qc_longitudinal_context_v1.tsv   de1954adf09463c8...
  mci_survival_manual_qc_form_v1.tsv                   9ad2db21c2067e2a...
  mci_survival_manual_qc_sampling_summary_

## 13. Build 2 summary

In [14]:
print("=" * 84)
print("BUILD 2 — MANUAL QC PACKET (v1)")
print("=" * 84)
print(f"Assertions            : {sum(a['passed'] for a in ASSERTIONS)} passed / "
      f"{sum(not a['passed'] for a in ASSERTIONS)} failed  (of {len(ASSERTIONS)})")
print(f"Seed                  : {RANDOM_SEED} (sampling verified reproducible)")
print(f"Unique participants   : {len(SELECTED)}  (before dedup {n_before_dedup}, "
      f"overlap {len(overlap_rids)})")
print(f"Context rows          : {len(qc_context)}  ({int((qc_context.record_type=='diagnosis').sum())} dx, "
      f"{int((qc_context.record_type=='plasma').sum())} plasma)")
print(f"Blank human fields    : {len(HUMAN_FIELDS)} x {len(qc_form)} rows")
print()
print(f"MANDATORY pre-anchor-dementia cases: {len(MANDATORY)}/{len(MANDATORY)} included")
print(f"  reach survival cohort : {len(m_fu)}  (events {len(m_ev)})")
print(f"  reach primary cohort  : {len(m_pr)}  (events {len(m_pr_ev)} -> "
      f"RID {sorted(int(r) for r in m_pr_ev['RID'])})")
print()
print("Frozen v1 cohort UNCHANGED: A=%d B=%d C=%d — no participant excluded by this build."
      % (N_ANCHOR, N_FU, N_PRIMARY))
print("=" * 84)

BUILD 2 — MANUAL QC PACKET (v1)
Assertions            : 34 passed / 0 failed  (of 34)
Seed                  : 20260711 (sampling verified reproducible)
Unique participants   : 46  (before dedup 48, overlap 2)
Context rows          : 362  (288 dx, 74 plasma)
Blank human fields    : 13 x 46 rows

MANDATORY pre-anchor-dementia cases: 8/8 included
  reach survival cohort : 5  (events 2)
  reach primary cohort  : 4  (events 1 -> RID [467])

Frozen v1 cohort UNCHANGED: A=535 B=410 C=401 — no participant excluded by this build.
